In [1]:

from pathlib import Path
import re, json, shutil, subprocess
import os

In [5]:

def _natural_key(s: str):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

def renumber_files_macname(
    folder_path,
    dry_run=True,
    digits=2,
    skip_hidden=True,
):
    folder = Path(folder_path)

    files = []
    for f in folder.iterdir():
        if not f.is_file():
            continue
        if skip_hidden and f.name.startswith("."):
            continue
        files.append(f)

    def cleaned_for_sort(name: str) -> str:
        # 排序时忽略已有编号前缀
        return re.sub(r"^\d+_", "", name)

    files.sort(key=lambda p: _natural_key(cleaned_for_sort(p.name)))

    print("\n=== ORDER (filtered, ignore old prefix) ===")
    for f in files:
        print(cleaned_for_sort(f.name))

    plan = []
    for idx, f in enumerate(files, start=1):
        cleaned = cleaned_for_sort(f.name)
        new_name = f"{idx:0{digits}d}_{cleaned}"
        plan.append((f, new_name))

    print("\n=== RENAME PLAN ===")
    for f, new_name in plan:
        print(f"{f.name}  →  {new_name}")

    if dry_run:
        print("\n[DRY RUN] No files were renamed.")
        return

    # 两步改名避免撞名
    temp_plan = []
    for i, (f, new_name) in enumerate(plan, start=1):
        tmp_name = f".__tmp__{i:05d}__{f.name}"
        tmp_path = folder / tmp_name
        f.rename(tmp_path)
        temp_plan.append((tmp_path, folder / new_name))

    for tmp_path, final_path in temp_plan:
        tmp_path.rename(final_path)

    print("\nRenaming complete.")

# renumber_files_macname("name", dry_run=True)
renumber_files_macname("name", dry_run=False)


=== ORDER (filtered, ignore old prefix) ===
第1集_天命世无双.m4a
第2集_请君入棋局.m4a
第3集_折翎亦藏锋.m4a
第4集_丹墀铸九鼎.m4a
第5集_一梦烟火空.m4a
第6集_曲终棋散时.m4a
第7集_为君射天狼.m4a
第8集_蛇蝎美人蛊.m4a
第9集_风月初相授.m4a
第10集_勘破局中局.m4a
第11集_请入相思瓮.m4a
第12集_龙袍又加身.m4a
第13集_秉烛待归人.m4a
第14集_风雪遇狼王.m4a
第15集_血肉生枯骨.m4a
第16集_长愿不相离.m4a

=== RENAME PLAN ===
第1集_天命世无双.m4a  →  01_第1集_天命世无双.m4a
第2集_请君入棋局.m4a  →  02_第2集_请君入棋局.m4a
第3集_折翎亦藏锋.m4a  →  03_第3集_折翎亦藏锋.m4a
第4集_丹墀铸九鼎.m4a  →  04_第4集_丹墀铸九鼎.m4a
第5集_一梦烟火空.m4a  →  05_第5集_一梦烟火空.m4a
第6集_曲终棋散时.m4a  →  06_第6集_曲终棋散时.m4a
第7集_为君射天狼.m4a  →  07_第7集_为君射天狼.m4a
第8集_蛇蝎美人蛊.m4a  →  08_第8集_蛇蝎美人蛊.m4a
第9集_风月初相授.m4a  →  09_第9集_风月初相授.m4a
第10集_勘破局中局.m4a  →  10_第10集_勘破局中局.m4a
第11集_请入相思瓮.m4a  →  11_第11集_请入相思瓮.m4a
第12集_龙袍又加身.m4a  →  12_第12集_龙袍又加身.m4a
第13集_秉烛待归人.m4a  →  13_第13集_秉烛待归人.m4a
第14集_风雪遇狼王.m4a  →  14_第14集_风雪遇狼王.m4a
第15集_血肉生枯骨.m4a  →  15_第15集_血肉生枯骨.m4a
第16集_长愿不相离.m4a  →  16_第16集_长愿不相离.m4a

Renaming complete.


In [36]:

def add_to_prefix_number(folder_path, x, dry_run=True):
    folder = Path(folder_path)

    print("\n=== RENAME PLAN ===")

    for f in folder.iterdir():
        if not f.is_file():
            continue
        
        # 跳过 DS_Store
        if ".DS_Store" in f.name:
            continue

        # 匹配开头数字_
        match = re.match(r'^(\d+)_', f.name)
        if not match:
            continue  # 没有数字前缀就跳过

        old_number = int(match.group(1))
        new_number = old_number + x

        # 保持原位数（比如 01 → 03）
        width = len(match.group(1))
        new_prefix = f"{new_number:0{width}d}_"

        # 替换旧前缀
        new_name = re.sub(r'^\d+_', new_prefix, f.name)

        print(f"{f.name}  →  {new_name}")

        if not dry_run:
            f.rename(folder / new_name)

    if dry_run:
        print("\n[DRY RUN] No files renamed.")
    else:
        print("\nRenaming complete.")

add_to_prefix_number("add", x=3, dry_run=True)
# add_to_prefix_number("add", x=3, dry_run=False)


=== RENAME PLAN ===
30_小剧场_雨后风荷.m4a  →  33_小剧场_雨后风荷.m4a
58_花絮1.m4a  →  61_花絮1.m4a
52_800w福利_与君书_手书.m4a  →  55_800w福利_与君书_手书.m4a
27_小剧场_假神仙.m4a  →  30_小剧场_假神仙.m4a
31_冬至特别语音.m4a  →  34_冬至特别语音.m4a
47_500w特别剧场_罄露_情动.m4a  →  50_500w特别剧场_罄露_情动.m4a
59_花絮2.m4a  →  62_花絮2.m4a
35_元宵特别语音.m4a  →  38_元宵特别语音.m4a
53_800w福利_与君书_批注.m4a  →  56_800w福利_与君书_批注.m4a
45_500w特别剧场_罄露_同榻.m4a  →  48_500w特别剧场_罄露_同榻.m4a
37_祝苏大人生辰康乐.m4a  →  40_祝苏大人生辰康乐.m4a
65_特别花絮.m4a  →  68_特别花絮.m4a
43_300w福利_得闲_断袖.m4a  →  46_300w福利_得闲_断袖.m4a
56_800w福利_与君书_纸条.m4a  →  59_800w福利_与君书_纸条.m4a
49_500w特别剧场_罄露_邀吻.m4a  →  52_500w特别剧场_罄露_邀吻.m4a
28_小剧场_冯党苏党？.m4a  →  31_小剧场_冯党苏党？.m4a
32_元旦特别语音.m4a  →  35_元旦特别语音.m4a
54_800w福利_与君书_日记.m4a  →  57_800w福利_与君书_日记.m4a
26_小剧场_小爷难缠.m4a  →  29_小剧场_小爷难缠.m4a
64_花絮7.m4a  →  67_花絮7.m4a
25_小剧场_鸳鸯腿.m4a  →  28_小剧场_鸳鸯腿.m4a
38_100W福利音_小憩.m4a  →  41_100W福利音_小憩.m4a
60_花絮3.m4a  →  63_花絮3.m4a
33_除夕特别剧场_“朱”你新年快乐.m4a  →  36_除夕特别剧场_“朱”你新年快乐.m4a
61_花絮4.m4a  →  64_花絮4.m4a
50_800w福利_与君书_便笺.m4a  →  53_800w福利_与君书_便笺.m4a
48_

# converter


## 目录示例

### 单本共用封面

```python
某书_某作者_甲/播音_乙/播音/
├─ audio/
│  ├─ 01_第1集.m4a
│  ├─ 02_第2集.m4a
│  └─ 03_第3集.m4a
└─ cover.jpg
```

输出：

```python
某书_某作者_甲/播音_乙/播音/
└─ 某书_某作者.m4b
```

---

### 多 part，每个 part 不同封面

```python
魔道祖师_墨香铜臭_蓝忘机/魏超_魏无羡/路知行/
├─ audio/
│  ├─ 魔道祖师_第1季/
│  │  ├─ 01_第1集.m4a
│  │  ├─ 02_第2集.m4a
│  │  └─ cover.jpg
│  ├─ 魔道祖师_第2季/
│  │  ├─ 01_第1集.m4a
│  │  └─ cover.jpg
│  └─ 魔道祖师_第3季/
│     ├─ 01_第1集.m4a
│     └─ cover.jpg
└─ cover.jpg
```

输出：

```python
魔道祖师_墨香铜臭_蓝忘机/魏超_魏无羡/路知行/
├─ 魔道祖师_第1季.m4b
├─ 魔道祖师_第2季.m4b
└─ 魔道祖师_第3季.m4b
```


In [6]:
from pathlib import Path
import re
import json
import shutil
import subprocess
import tempfile

ROOT_DIR = Path(".").resolve()

def which(cmd):
    p = shutil.which(cmd)
    if not p:
        raise RuntimeError(f"未找到 {cmd}，请先安装：brew install ffmpeg")
    return p

FFMPEG = which("ffmpeg")
FFPROBE = which("ffprobe")


def sort_key(p: Path):
    name = p.stem
    m = re.match(r"^\s*(\d+)", name)
    if m:
        return (0, int(m.group(1)))
    return (1, name.lower())


def clean_chapter_title(p: Path):
    name = p.stem
    name = re.sub(r"^\s*\d+\s*[-_. ]+\s*", "", name)
    return name.strip()


def duration_seconds(path: Path) -> float:
    cmd = [
        FFPROBE, "-v", "error",
        "-of", "json",
        "-show_format",
        "-show_streams",
        str(path)
    ]
    out = subprocess.check_output(cmd)
    data = json.loads(out)

    def is_valid_number(x):
        try:
            return x is not None and x != "N/A" and float(x) > 0
        except:
            return False

    fmt = data.get("format", {})
    dur = fmt.get("duration")
    if is_valid_number(dur):
        return float(dur)

    streams = data.get("streams", [])
    a0 = None
    for s in streams:
        if s.get("codec_type") == "audio":
            a0 = s
            break

    if a0:
        sdur = a0.get("duration")
        if is_valid_number(sdur):
            return float(sdur)

        duration_ts = a0.get("duration_ts")
        sample_rate = a0.get("sample_rate")
        if duration_ts not in (None, "N/A") and sample_rate not in (None, "N/A"):
            return float(duration_ts) / float(sample_rate)

    raise RuntimeError(f"无法获取时长：{path}")


def escape_ffmeta_value(s: str) -> str:
    return (
        str(s)
        .replace("\\", "\\\\")
        .replace("=", r"\=")
        .replace(";", r"\;")
        .replace("#", r"\#")
        .replace("\n", " ")
    )


def find_cover_in_dir(folder: Path):
    candidates = [
        folder / "cover.jpg",
        folder / "cover.jpeg",
        folder / "cover.png",
        folder / "cover.JPG",
        folder / "cover.JPEG",
        folder / "cover.PNG",
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


def parse_album_author(book_dir_name: str):
    parts = book_dir_name.split("_")
    if len(parts) < 2:
        raise ValueError(f"文件夹名不符合要求，至少应有两段：{book_dir_name}")
    album = "_".join(parts[:2])
    author = "_".join(parts[2:]) if len(parts) > 2 else parts[1]
    return album, author


def collect_parts(audio_dir: Path):
    """
    - 若 audio/ 下有子文件夹：每个子文件夹一个 part
    - 否则 audio/ 下所有 m4a 作为一个整体
    """
    subdirs = sorted([p for p in audio_dir.iterdir() if p.is_dir()], key=lambda p: p.name.lower())
    root_audio = sorted(audio_dir.glob("*.m4a"), key=sort_key)

    if subdirs:
        parts = []
        for d in subdirs:
            files = sorted(d.glob("*.m4a"), key=sort_key)
            if files:
                parts.append({
                    "part_name": d.name,
                    "part_dir": d,
                    "files": files,
                })
        return parts, True

    if root_audio:
        return [{
            "part_name": None,
            "part_dir": audio_dir,
            "files": root_audio,
        }], False

    return [], False


def build_one_m4b(files, final_m4b: Path, title_value: str, album_value: str, author_value: str, cover_path: Path):
    with tempfile.TemporaryDirectory() as td:
        td = Path(td)

        concat_list = td / "concat_list.txt"
        merged_tmp = td / "merged_tmp.m4b"
        ffmeta = td / "chapters.ffmeta"

        with concat_list.open("w", encoding="utf-8") as f:
            for p in files:
                abs_path = p.resolve().as_posix().replace("'", r"'\''")
                f.write(f"file '{abs_path}'\n")

        merge_cmd = [
            FFMPEG, "-y",
            "-f", "concat", "-safe", "0",
            "-i", str(concat_list),
            "-c", "copy",
            str(merged_tmp)
        ]
        subprocess.run(merge_cmd, check=True)

        durations = [duration_seconds(p) for p in files]

        start_ms = 0
        lines = []
        lines.append(";FFMETADATA1")
        lines.append(f"title={escape_ffmeta_value(title_value)}")
        lines.append(f"artist={escape_ffmeta_value(author_value)}")
        lines.append(f"album={escape_ffmeta_value(album_value)}")

        for p, d in zip(files, durations):
            end_ms = start_ms + int(round(d * 1000))
            chapter_title = clean_chapter_title(p)

            lines.append("[CHAPTER]")
            lines.append("TIMEBASE=1/1000")
            lines.append(f"START={start_ms}")
            lines.append(f"END={end_ms}")
            lines.append(f"title={escape_ffmeta_value(chapter_title)}")

            start_ms = end_ms

        ffmeta.write_text("\n".join(lines) + "\n", encoding="utf-8")

        final_cmd = [
            FFMPEG, "-y",
            "-i", str(merged_tmp),
            "-i", str(cover_path),
            "-f", "ffmetadata", "-i", str(ffmeta),
            "-map", "0:a:0",
            "-map", "1:v:0",
            "-map_metadata", "2",
            "-map_chapters", "2",
            "-c:a", "copy",
            "-c:v", "mjpeg",
            "-disposition:v:0", "attached_pic",
            str(final_m4b)
        ]
        subprocess.run(final_cmd, check=True)


def process_one_book(book_dir: Path):
    audio_dir = book_dir / "audio"
    if not audio_dir.exists() or not audio_dir.is_dir():
        print(f"⏭ Skip: {book_dir.name}  (missing audio/)")
        return []

    album, author = parse_album_author(book_dir.name)
    book_cover = find_cover_in_dir(book_dir)

    parts, has_parts = collect_parts(audio_dir)
    if not parts:
        print(f"⏭ Skip: {book_dir.name}  (audio/ has no .m4a)")
        return []

    outputs = []
    print(f"\n📚 Processing: {book_dir.name}")
    print(f"   ALBUM : {album}")
    print(f"   AUTHOR: {author}")

    if has_parts:
        print(f"   Mode  : multi-part ({len(parts)} parts)")
        for part in parts:
            part_name = part["part_name"]
            part_dir = part["part_dir"]
            files = part["files"]

            part_cover = find_cover_in_dir(part_dir)
            cover_to_use = part_cover or book_cover

            if cover_to_use is None:
                print(f"   ⏭ Skip part: {part_name}  (missing part cover and book cover)")
                continue

            final_m4b = book_dir / f"{part_name}.m4b"

            print(f"   ▶ Part : {part_name}")
            print(f"      Cover: {cover_to_use.relative_to(book_dir)}")
            print(f"      Files: {len(files)}")

            build_one_m4b(
                files=files,
                final_m4b=final_m4b,
                title_value=part_name,
                album_value=album,
                author_value=author,
                cover_path=cover_to_use
            )
            outputs.append(final_m4b)

    else:
        files = parts[0]["files"]
        if book_cover is None:
            print(f"⏭ Skip: {book_dir.name}  (single-book mode requires book cover)")
            return []

        final_m4b = book_dir / f"{album}.m4b"

        print(f"   Mode  : single-book")
        print(f"   Cover : {book_cover.name}")
        print(f"   Files : {len(files)}")

        build_one_m4b(
            files=files,
            final_m4b=final_m4b,
            title_value=album,
            album_value=album,
            author_value=author,
            cover_path=book_cover
        )
        outputs.append(final_m4b)

    return outputs


book_dirs = sorted([p for p in ROOT_DIR.iterdir() if p.is_dir()], key=lambda p: p.name.lower())

all_outputs = []

for book_dir in book_dirs:
    try:
        outs = process_one_book(book_dir)
        all_outputs.extend(outs)
    except Exception as e:
        print(f"❌ Failed: {book_dir.name}")
        print("   ", e)

print("\n✅ Done. Output file(s):")
for p in all_outputs:
    print(" -", p.resolve())

⏭ Skip: .git  (missing audio/)
⏭ Skip: name  (missing audio/)

📚 Processing: 锁帝翎_崖生_萧独:张福正_萧翎:文森
   ALBUM : 锁帝翎_崖生
   AUTHOR: 萧独:张福正_萧翎:文森
   Mode  : single-book
   Cover : cover.jpg
   Files : 22


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.3.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1_4 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60.  8.100 / 60.  8.100
  libavcodec     62. 11.100 / 62. 11.100
  libavformat    62.  3.100 / 62.  3.100
  libavdevice    62.  1.100 / 62.  1.100
  libavfilter    11.  4.100 / 11.  4.100
  libswscale      9.  1.100 /  9.  1.100
  libswresample   6.  1.100 /  6.  1.100
[mov,mp4,m4a,3gp,3g2,mj2 @ 0xa9cc14280] Detected creation time before 1970, parsing as unix timestamp.
Input #0, concat, from '/var/folders/cd/9k1mgrwj149fm1s888m3g8l00000gn/T/tmpl61gjnmu/concat_list.txt


✅ Done. Output file(s):
 - /Users/jijiko/Desktop/GitHub/audio-to-audiobook/锁帝翎_崖生_萧独:张福正_萧翎:文森/锁帝翎_崖生.m4b


[out#0/ipod @ 0x7f9010180] video:72KiB audio:334241KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 2.154828%
frame=    1 fps=0.2 q=7.0 Lsize=  341517KiB time=00:00:00.04 bitrate=69942737.8kbits/s speed=0.00812x elapsed=0:00:04.92    
